# 03 - Regularización y diagnóstico bias/variance

Se comparan una red sin regularización y otra con L2, Dropout y Early Stopping usando una muestra estratificada de 12.000 imágenes de Train. Dev monitorea la generalización. Test no se utiliza en estos ajustes.

## 1. Preparar datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

In [ ]:
(x_all, y_all), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_dev, y_train, y_dev = train_test_split(
    x_all, y_all, test_size=0.20, random_state=SEED, stratify=y_all
)
x_train = x_train.astype("float32") / 255.0
x_dev = x_dev.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print("Train:", x_train.shape, y_train.shape)
print("Dev:", x_dev.shape, y_dev.shape)
print("Test:", x_test.shape, y_test.shape)
print("Rango de píxeles:", x_train.min(), x_train.max())

## 2. Muestra estratificada

In [ ]:
x_train_reg, _, y_train_reg, _ = train_test_split(
    x_train, y_train, train_size=12000, random_state=SEED, stratify=y_train
)
print(x_train_reg.shape, y_train_reg.shape)

## 3. Modelo sin regularización

In [ ]:
def crear_sin_reg():
    m = keras.Sequential([
        layers.Input(shape=(28, 28)), layers.Flatten(),
        layers.Dense(512, activation="relu"),
        layers.Dense(256, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    m.compile(optimizer=keras.optimizers.Adam(0.001),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
modelo_sin_reg = crear_sin_reg()
hist_sin_reg = modelo_sin_reg.fit(
    x_train_reg, y_train_reg, validation_data=(x_dev, y_dev),
    epochs=15, batch_size=64, verbose=0
)

## 4. Modelo regularizado: L2 + Dropout + Early Stopping

In [ ]:
def crear_regularizado():
    m = keras.Sequential([
        layers.Input(shape=(28, 28)), layers.Flatten(),
        layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.30),
        layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.30),
        layers.Dense(10, activation="softmax")
    ])
    m.compile(optimizer=keras.optimizers.Adam(0.001),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

parada = callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
modelo_reg = crear_regularizado()
hist_reg = modelo_reg.fit(
    x_train_reg, y_train_reg, validation_data=(x_dev, y_dev),
    epochs=20, batch_size=64, callbacks=[parada], verbose=0
)
print("Épocas ejecutadas con Early Stopping:", len(hist_reg.history["loss"]))

## 5. Comparar curvas y métricas Train/Dev

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(hist_sin_reg.history["loss"], label="Train sin reg.")
ax.plot(hist_sin_reg.history["val_loss"], label="Dev sin reg.")
ax.plot(hist_reg.history["loss"], label="Train regularizado")
ax.plot(hist_reg.history["val_loss"], label="Dev regularizado")
ax.set(xlabel="Época", ylabel="Loss", title="Efecto de la regularización")
ax.grid(True, alpha=.3); ax.legend(); plt.show()

def resumen(nombre, modelo, historia):
    tr_loss, tr_acc = modelo.evaluate(x_train_reg, y_train_reg, verbose=0)
    dv_loss, dv_acc = modelo.evaluate(x_dev, y_dev, verbose=0)
    return {"modelo": nombre, "train_loss": tr_loss, "dev_loss": dv_loss,
            "train_accuracy": tr_acc, "dev_accuracy": dv_acc,
            "brecha_accuracy": tr_acc-dv_acc, "epocas": len(historia.history["loss"])}
display(pd.DataFrame([
    resumen("Sin regularización", modelo_sin_reg, hist_sin_reg),
    resumen("L2 + Dropout + Early Stopping", modelo_reg, hist_reg)
]))

## 6. Bias/variance: modelo pequeño

In [ ]:
keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
modelo_pequeno = keras.Sequential([
    layers.Input(shape=(28, 28)), layers.Flatten(),
    layers.Dense(16, activation="relu"), layers.Dense(10, activation="softmax")
])
modelo_pequeno.compile(optimizer=keras.optimizers.Adam(0.001),
                       loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_pequeno = modelo_pequeno.fit(
    x_train, y_train, validation_data=(x_dev, y_dev),
    epochs=2, batch_size=64, verbose=0
)
display(pd.DataFrame([
    {"modelo": n, "train_accuracy": h.history["accuracy"][-1],
     "dev_accuracy": h.history["val_accuracy"][-1],
     "brecha": h.history["accuracy"][-1] - h.history["val_accuracy"][-1]}
    for n, h in [("Pequeño", hist_pequeno), ("Sin regularización", hist_sin_reg),
                 ("Regularizado", hist_reg)]
]))

**Análisis:** Train y Dev bajos sugieren underfitting/bias alto; Train alto y Dev claramente inferior puede indicar overfitting/variance alta. Comenta si L2, Dropout y Early Stopping redujeron la brecha, cuántas épocas ejecutó la parada temprana y cuál configuración generalizó mejor según Dev.